In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [19]:
df = pd.read_csv('/content/heart_disease_uci.csv')

In [20]:
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:-1],df.iloc[:,-1],test_size=0.2,random_state=2)

### Preprocessing: One-Hot Encoding Categorical Features

Before training the models, we need to convert the categorical features (like 'sex', 'dataset', 'cp', etc.) into a numerical format. One-hot encoding is a suitable method for this, where each category is converted into a new binary column. We also drop the 'id' column as it is an identifier and not useful for prediction.

In [22]:
# Drop the 'id' column from X_train and X_test
X_train = X_train.drop('id', axis=1)
X_test = X_test.drop('id', axis=1)

# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object', 'bool']).columns

# Apply one-hot encoding to training data
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)

# Apply one-hot encoding to test data
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - this is crucial to ensure both train and test sets have the same columns
# after one-hot encoding, especially if some categories are not present in both.
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Display the first few rows of the encoded training data
print("Encoded X_train head:")
display(X_train_encoded.head())

print("\nShape of encoded X_train:", X_train_encoded.shape)
print("Shape of encoded X_test:", X_test_encoded.shape)


Encoded X_train head:


,age,trestbps,chol,thalch,oldpeak,ca,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,...,cp_non-anginal,cp_typical angina,fbs_True,restecg_normal,restecg_st-t abnormality,exang_True,slope_flat,slope_upsloping,thal_normal,thal_reversable defect
400,48,100.0,NaN,100.0,0.0,NaN,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
107,57,128.0,229.0,150.0,0.4,1.0,True,False,False,False,...,True,False,False,False,False,False,True,False,False,True
566,52,140.0,404.0,124.0,2.0,NaN,True,True,False,False,...,False,False,False,True,False,True,True,False,False,False
580,65,170.0,263.0,112.0,2.0,NaN,True,True,False,False,...,False,False,True,True,False,True,True,False,False,False
372,44,130.0,215.0,135.0,0.0,NaN,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False



Shape of encoded X_train: (736, 21)
Shape of encoded X_test: (184, 21)


### Handling Missing Numerical Values

Before retraining, we need to address any remaining missing values (NaNs) in the numerical columns. We will use `SimpleImputer` to fill these with the mean of their respective columns.

In [23]:
from sklearn.impute import SimpleImputer

# Initialize Imputer for numerical columns
imputer = SimpleImputer(strategy='mean')

# Fit on training data and transform both training and test data
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_encoded), columns=X_train_encoded.columns, index=X_train_encoded.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_encoded), columns=X_test_encoded.columns, index=X_test_encoded.index)

print("X_train_imputed head:")
display(X_train_imputed.head())

print("\nNumber of NaNs in X_train_imputed:", X_train_imputed.isnull().sum().sum())
print("Number of NaNs in X_test_imputed:", X_test_imputed.isnull().sum().sum())

X_train_imputed head:


,age,trestbps,chol,thalch,oldpeak,ca,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,...,cp_non-anginal,cp_typical angina,fbs_True,restecg_normal,restecg_st-t abnormality,exang_True,slope_flat,slope_upsloping,thal_normal,thal_reversable defect
400,48.0,100.0,201.107843,100.0,0.0,0.692607,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
107,57.0,128.0,229.000000,150.0,0.4,1.000000,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
566,52.0,140.0,404.000000,124.0,2.0,0.692607,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
580,65.0,170.0,263.000000,112.0,2.0,0.692607,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
372,44.0,130.0,215.000000,135.0,0.0,0.692607,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0



Number of NaNs in X_train_imputed: 0
Number of NaNs in X_test_imputed: 0


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [25]:
clf1 = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
clf2 = DecisionTreeClassifier()

In [26]:
# Retrain models with imputed data
clf1.fit(X_train_imputed, y_train)
clf2.fit(X_train_imputed, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


DecisionTreeClassifier()

### Retrain Models with Preprocessed Data

Now that the categorical features have been converted to numerical format, we can retrain the Logistic Regression and Decision Tree Classifier models.

In [28]:
y_pred1 = clf1.predict(X_test_imputed)
y_pred2 = clf2.predict(X_test_imputed)

In [29]:
from sklearn.metrics import accuracy_score,confusion_matrix
print("Accuracy of Logistic Regression",accuracy_score(y_test,y_pred1))
print("Accuracy of Decision Trees",accuracy_score(y_test,y_pred2))

Accuracy of Logistic Regression 0.5434782608695652
Accuracy of Decision Trees 0.4891304347826087


In [30]:
confusion_matrix(y_test,y_pred1)

array([[64, 11,  2,  1,  1],
       [20, 26,  8,  6,  0],
       [ 0,  8,  4, 10,  0],
       [ 2,  6,  4,  6,  0],
       [ 0,  2,  2,  1,  0]])

In [32]:
print("Logistic Regression Confusion Matrix\n")
pd.DataFrame(confusion_matrix(y_test,y_pred1),columns=list(range(0,5)))

Logistic Regression Confusion Matrix



,0,1,2,3,4
0,64,11,2,1,1
1,20,26,8,6,0
2,0,8,4,10,0
3,2,6,4,6,0
4,0,2,2,1,0


In [36]:
print("Decision Tree Confusion Matrix\n")
pd.DataFrame(confusion_matrix(y_test,y_pred2),columns=list(range(0,5)))

Decision Tree Confusion Matrix



,0,1,2,3,4
0,58,12,1,7,1
1,18,22,9,11,0
2,1,7,5,7,2
3,2,6,4,5,1
4,0,0,2,3,0


In [37]:

result = pd.DataFrame()
result['Actual Label'] = y_test
result['Logistic Regression Prediction'] = y_pred1
result['Decision Tree Prediction'] = y_pred2

In [38]:
result.sample(10)


,Actual Label,Logistic Regression Prediction,Decision Tree Prediction
868,3,2,1
650,4,2,3
212,0,0,0
636,2,3,0
387,0,0,1
854,3,3,1
65,2,3,4
842,1,3,2
809,1,0,3
193,2,3,1


In [39]:
from sklearn.metrics import recall_score,precision_score,f1_score

In [41]:
print("For Logistic regression Model")
print("-"*50)
cdf = pd.DataFrame(confusion_matrix(y_test,y_pred1),columns=list(range(0,5)))
print(cdf)
print("-"*50)
print("Precision - ",precision_score(y_test,y_pred1, average='weighted'))
print("Recall - ",recall_score(y_test,y_pred1, average='weighted'))
print("F1 score - ",f1_score(y_test,y_pred1, average='weighted'))

For Logistic regression Model
--------------------------------------------------
    0   1  2   3  4
0  64  11  2   1  1
1  20  26  8   6  0
2   0   8  4  10  0
3   2   6  4   6  0
4   0   2  2   1  0
--------------------------------------------------
Precision -  0.5278514127096171
Recall -  0.5434782608695652
F1 score -  0.5338521809087411


In [43]:
print("For DT Model")
print("-"*50)
cdf = pd.DataFrame(confusion_matrix(y_test,y_pred2),columns=list(range(0,5)))
print(cdf)
print("-"*50)
print("Precision - ",precision_score(y_test,y_pred2, average='weighted'))
print("Recall - ",recall_score(y_test,y_pred2, average='weighted'))
print("F1 score - ",f1_score(y_test,y_pred2, average='weighted'))

For DT Model
--------------------------------------------------
    0   1  2   3  4
0  58  12  1   7  1
1  18  22  9  11  0
2   1   7  5   7  2
3   2   6  4   5  1
4   0   0  2   3  0
--------------------------------------------------
Precision -  0.5111438823279711
Recall -  0.4891304347826087
F1 score -  0.4962966740800816


In [44]:
precision_score(y_test,y_pred1,average=None)

array([0.74418605, 0.49056604, 0.2       , 0.25      , 0.        ])

In [45]:
precision_score(y_test,y_pred2,average=None)

array([0.73417722, 0.46808511, 0.23809524, 0.15151515, 0.        ])

In [46]:
recall_score(y_test,y_pred2,average=None)

array([0.73417722, 0.36666667, 0.22727273, 0.27777778, 0.        ])